# Extending PyTorch: custom layers, losses, and autograd Functions

_PyTorch_

**Write your own nn.Module layer with learnable nn.Parameters, a custom loss function, and — when you need a hand-written gradient — a torch.autograd.Function verified by gradcheck.**

PyTorch is extensible at three layers, and the trick is knowing how far down you must go.

       Most of the time you never touch the backward pass. If your idea is built from differentiable
       torch operations &mdash; matrix multiplies, additions, exponentials, comparisons that route gradients &mdash;
       then writing the forward is enough. Autograd recorded every op into a graph and will replay it
       in reverse to get exact gradients. A custom layer and a custom loss are exactly this: you supply the
       forward formula and learnable nn.Parameters, and the gradient comes for free.

---

This notebook is a practice scaffold for the **Extending PyTorch: custom layers, losses, and autograd Functions** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)                          # reproducible weights and data

# ============================================================
# 1. A CUSTOM LAYER: a from-scratch Linear.
#    Weights are nn.Parameter (so they TRAIN); a counter is a
#    register_buffer (non-learnable state that still saves/moves).
# ============================================================
class MyLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()                    # MUST be first
        # nn.Parameter -> registered + requires_grad=True -> the optimizer trains it.
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias   = nn.Parameter(torch.zeros(out_features))
        # A buffer: NON-learnable state. Moves with .to(device), saved in state_dict,
        # but never updated by the optimizer.
        self.register_buffer("call_count", torch.zeros(1))

    def forward(self, x):
        self.call_count += 1                  # buffer bookkeeping
        return x @ self.weight.t() + self.bias    # pure differentiable ops -> autograd backward for free

layer = MyLinear(4, 3)
print("params (trained):", [name for name, _ in layer.named_parameters()])
#   -> ['weight', 'bias']                     (call_count is NOT here)
print("buffers (not trained):", [name for name, _ in layer.named_buffers()])
#   -> ['call_count']
print("state_dict keys:", list(layer.state_dict().keys()))
#   -> ['weight', 'bias', 'call_count']       (buffer IS saved)

# ============================================================
# 2. A CUSTOM LOSS: just a function returning a scalar.
#    Built from differentiable torch ops -> autograd handles backward.
# ============================================================
def my_mse(pred, target):
    return ((pred - target) ** 2).mean()

pred   = torch.randn(8, 3)
target = torch.randn(8, 3)
print("my_mse vs nn.MSELoss:",
      float(my_mse(pred, target)),
      float(nn.MSELoss()(pred, target)))      # identical

# ============================================================
# 3. A CUSTOM AUTOGRAD FUNCTION: square, with a HAND-WRITTEN gradient.
#    Use this when you need to own the backward (non-standard /
#    more efficient gradient, or to wrap non-PyTorch code).
# ============================================================
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)              # backward needs x for d/dx(x^2)=2x
        return x * x

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x            # chain rule: dL/dx = dL/dy * 2x

square = Square.apply                          # call via .apply, NOT Square.forward

# Verify the backward math. gradcheck needs float64 inputs + requires_grad.
gx = torch.randn(5, dtype=torch.float64, requires_grad=True)
ok = torch.autograd.gradcheck(square, (gx,), eps=1e-6, atol=1e-4)
print("gradcheck passed:", ok)                # -> True (our 2x backward is correct)

# ============================================================
# 4. TRAIN the custom layer with the custom loss for a few steps.
# ============================================================
torch.manual_seed(1)
X = torch.randn(64, 4)
true_W = torch.randn(3, 4); true_b = torch.randn(3)
Y = X @ true_W.t() + true_b                    # the target the layer should learn

model = MyLinear(4, 3)
opt = torch.optim.Adam(model.parameters(), lr=0.1)   # only weight & bias are passed
for step in range(200):
    opt.zero_grad()                            # clear old grads (they accumulate!)
    loss = my_mse(model(X), Y)                 # custom loss; autograd backprops it
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")
print("final loss:", round(my_mse(model(X), Y).item(), 5))   # near 0
print("forward calls counted by buffer:", int(model.call_count.item()))

## Visualize the data & results

_Is the hand-written backward of a custom autograd Function correct? For Square (f(x)=x^2), plot the analytic gradient our backward returns (2x) against a numerical finite-difference gradient across x — gradcheck's idea, by hand._

In [ ]:
import numpy as np

xs = np.linspace(-3, 3, 31)

# Analytic gradient our custom Square.backward returns:  d/dx (x^2) = 2x
analytic = 2 * xs

# Numerical gradient via central finite differences (what gradcheck does):
#   f'(x) ~= (f(x+h) - f(x-h)) / (2h)
h = 1e-4
f = lambda x: x ** 2
numerical = (f(xs + h) - f(xs - h)) / (2 * h)

print("max abs difference:", np.max(np.abs(analytic - numerical)))   # ~1e-11
for xv, a, n in zip(xs[::5], analytic[::5], numerical[::5]):
    print(f"x={xv:+.1f}  analytic={a:+6.3f}  numerical={n:+6.3f}")

import matplotlib.pyplot as plt
plt.plot(xs, analytic, color="#4ea1ff", label="analytic backward 2x", linewidth=3)
plt.plot(xs, numerical, color="#ff7b72", label="numerical (finite diff)", linestyle="--")
plt.xlabel("x"); plt.ylabel("df/dx")
plt.title("Custom Function backward (2x) vs numerical gradient of f(x) = x^2")
plt.legend(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Write a MyLinear(nn.Module) from scratch: in __init__ hold self.W = nn.Parameter(torch.randn(out, in_) * 0.1) and self.b = nn.Parameter(torch.zeros(out)); in forward return x @ self.W.t() + self.b. Instantiate MyLinear(4, 3), run a (2, 4) input through it, and print the output shape. Use seed 0.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Subclass nn.Module, call super().__init__() first, and store weights as nn.Parameter. — _nn.Parameter registers the tensor so it lands in parameters() and the optimizer trains it._
- Implement the math with plain differentiable ops in forward. — _x @ W.t() + b is differentiable, so autograd builds the backward pass for free — no hand-written gradient._

**Answer:** import torch
import torch.nn as nn
torch.manual_seed(0)

class MyLinear(nn.Module):
    def __init__(self, in_, out):
        super().__init__()
        self.W = nn.Parameter(torch.randn(out, in_) * 0.1)
        self.b = nn.Parameter(torch.zeros(out))
    def forward(self, x):
        return x @ self.W.t() + self.b

layer = MyLinear(4, 3)
print(layer(torch.randn(2, 4)).shape)   # torch.Size([2, 3])

</details>

**Problem 2.** Type this in Colab. Build the famous pitfall: a layer that stores its weight as a plain tensor self.W = torch.randn(3, 4) instead of an nn.Parameter. Print list(layer.named_parameters()). Predict: is the weight in there? Then fix it with nn.Parameter and reprint.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print named_parameters() with the plain-tensor version. — _A bare tensor assigned to self is NOT registered, so it is missing from parameters() and the optimizer never trains it._
- Wrap it in nn.Parameter(...) and reprint. — _nn.Parameter registers it and sets requires_grad=True, so now it appears and will train._

**Answer:** class Bad(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = torch.randn(3, 4)        # plain tensor -> NOT registered
print(list(Bad().named_parameters()))     # []  -- empty! optimizer sees nothing

class Good(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Parameter(torch.randn(3, 4))
print([n for n, _ in Good().named_parameters()])   # ['W']

</details>

**Problem 3.** Type this in Colab. Write a custom loss as a plain function my_mse(pred, target) returning ((pred - target) ** 2).mean(). On pred = torch.randn(8, 3) and target = torch.randn(8, 3) (seed 0), print your loss next to nn.MSELoss()(pred, target) and confirm they match.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the loss from differentiable torch ops only. — _A loss is just a scalar-returning function; if it is differentiable, autograd backpropagates it with no extra code._
- Compare against the built-in nn.MSELoss. — _It confirms your hand-rolled formula matches PyTorch's reduction (mean over all elements)._

**Answer:** torch.manual_seed(0)
def my_mse(pred, target):
    return ((pred - target) ** 2).mean()

pred   = torch.randn(8, 3)
target = torch.randn(8, 3)
print(float(my_mse(pred, target)))            # e.g. 1.8945
print(float(nn.MSELoss()(pred, target)))      # identical value

</details>

**Problem 4.** Type this in Colab. Implement a torch.autograd.Function named Square for $f(x)=x^2$: in forward save x with ctx.save_for_backward(x) and return x * x; in backward return grad_output * 2 * x. Call it via Square.apply on x = torch.tensor([3.0], requires_grad=True), backprop, and print x.grad. Predict the value first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Define static forward(ctx, x) and backward(ctx, grad_output), calling the op via .apply. — _.apply records the op on the autograd graph; calling forward directly bypasses autograd._
- Predict the gradient: $\tfrac{d}{dx}x^2 = 2x$, so at $x=3$ it is $6$. — _Verifying the hand-written backward against the known derivative is the whole point of owning the gradient._

**Answer:** class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * x
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x

x = torch.tensor([3.0], requires_grad=True)
y = Square.apply(x)
y.backward()
print(x.grad)        # tensor([6.])  -- 2*x at x=3

</details>

**Problem 5.** Type this in Colab. Verify the Square Function's backward with torch.autograd.gradcheck. Make a float64 input with requires_grad=True and print the result. Then change the backward to a WRONG formula (grad_output * x) and show gradcheck now fails.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run torch.autograd.gradcheck(Square.apply, (x,)) on a .double() input. — _Finite differences need 64-bit headroom; on float32 gradcheck reports spurious mismatches._
- Break the backward (return grad_output * x) and rerun. — _gradcheck compares analytic vs numerical gradient and raises on a wrong derivative — catching the bug before training._

**Answer:** x = torch.randn(5, dtype=torch.float64, requires_grad=True)
print(torch.autograd.gradcheck(Square.apply, (x,)))   # True  -- correct backward

class BadSquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return x * x
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * x          # WRONG: should be 2*x
try:
    torch.autograd.gradcheck(BadSquare.apply, (x,))
except Exception as e:
    print("gradcheck FAILED:", type(e).__name__)   # raises -> bug caught

</details>

**Problem 6.** Type this in Colab. In an nn.Module, register a fixed lookup table with self.register_buffer("table", torch.arange(6.0).reshape(2, 3)). Print list(m.named_parameters()), list(m.named_buffers()), and list(m.state_dict().keys()). Predict which lists contain table.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use register_buffer for non-learnable state that must travel with the model. — _A buffer moves with model.to(device) and is saved in state_dict, but the optimizer ignores it._
- Predict: table is in buffers and state_dict, NOT in parameters. — _Only nn.Parameters show up in parameters(); a buffer is deliberately excluded so it is never trained._

**Answer:** class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("table", torch.arange(6.0).reshape(2, 3))

m = M()
print([n for n, _ in m.named_parameters()])   # []          -- not trained
print([n for n, _ in m.named_buffers()])      # ['table']   -- registered state
print(list(m.state_dict().keys()))            # ['table']   -- checkpointed

</details>

**Problem 7.** Type this in Colab. Show why nn.Parameter matters end-to-end. Train your MyLinear(4, 3) (from the first task) to fit Y = X @ true_W.t() + true_b for 200 Adam steps with your my_mse loss; print the loss every 50 steps. Use seeds so the run is reproducible.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass model.parameters() to torch.optim.Adam and run the train loop. — _Because W and b are nn.Parameters they are in parameters(), so the optimizer can update them._
- Call zero_grad() before each backward(). — _Gradients accumulate by default; clearing them keeps each step's update correct so the loss drops to near zero._

**Answer:** torch.manual_seed(1)
X = torch.randn(64, 4)
true_W, true_b = torch.randn(3, 4), torch.randn(3)
Y = X @ true_W.t() + true_b

model = MyLinear(4, 3)
opt = torch.optim.Adam(model.parameters(), lr=0.1)
for step in range(200):
    opt.zero_grad()
    loss = my_mse(model(X), Y)
    loss.backward(); opt.step()
    if step % 50 == 0:
        print(step, round(loss.item(), 4))
# 0  large; loss falls each block
print("final:", round(my_mse(model(X), Y).item(), 5))   # near 0.0

</details>

**Problem 8.** Type this in Colab. Demonstrate the non-differentiable pitfall. Take x = torch.tensor([1.7], requires_grad=True), compute y = torch.round(x), backprop y, and print x.grad. Predict the gradient before running, then explain in a comment why a custom Function would be needed for a usable gradient.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Backprop through torch.round and inspect x.grad. — _Rounding has zero derivative almost everywhere, so autograd hands back a zero gradient — no learning signal._
- Note the fix: a custom autograd.Function with a straight-through estimator. — _When you need a gradient through a non-differentiable step, you must hand-write the backward yourself._

**Answer:** x = torch.tensor([1.7], requires_grad=True)
y = torch.round(x)
y.backward()
print(x.grad)        # tensor([0.])  -- round has zero gradient -> no signal
# Fix: a torch.autograd.Function whose backward passes grad_output through
#      unchanged (a straight-through estimator).

</details>